In [ ]:
from tqdm import tqdm
import pandas as pd
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
import seaborn as sns


from itertools import cycle
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.calibration import calibration_curve
from sklearn.metrics import (accuracy_score, auc, classification_report, confusion_matrix, roc_auc_score, roc_curve)
from tensorflow import keras
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing import image
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from tqdm import tqdm

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### Load Data Paths
#### Read dataset and split paths from environment variables.

In [ ]:
dataset_dir = os.getenv("DATASET_DIR")
datasplit_dir = os.getenv("DATASPLITS_DIR")
models_dir = os.getenv("MODELS_DIR")

train_path = os.path.join(datasplit_dir, "mm_train.tsv")
test_path = os.path.join(datasplit_dir, "mm_test.tsv")

In [ ]:
df_train = pd.read_csv(train_path, sep = "\t")
df_test = pd.read_csv(test_path, sep = "\t")

print(df_test.shape)
print(df_test.columns)


In [ ]:
# Fit label encoders on training labels for both tasks
le_info = LabelEncoder()
le_info.fit(df_train["mm_info"])

le_human = LabelEncoder()
le_human.fit(df_train["mm_human"])

y_test_task1 = le_info.transform(df_test["mm_info"])
y_test_task2 = le_human.transform(df_test["mm_human"])

### Load Roberta model for text prediction

In [ ]:
# Path of both RoBERTa models
roberta_path_task1 = os.path.join(models_dir, "roberta-base_Task1")
roberta_path_task2 = os.path.join(models_dir, "roberta-base_Task2")

# Load RoBERTa model's for both tasks
tokenizer_roberta_task1 = RobertaTokenizer.from_pretrained(roberta_path_task1)
model_roberta_task1 = RobertaForSequenceClassification.from_pretrained(roberta_path_task1)

tokenizer_roberta_task2 = RobertaTokenizer.from_pretrained(roberta_path_task2)
model_roberta_task2 = RobertaForSequenceClassification.from_pretrained(roberta_path_task2)



### Load EfficientNetb0 for image prediction

In [ ]:
# Load weights
model_effnet = keras.models.load_model(f"{models_dir}/EfficientNetB0_finetuned.model.keras")

#Inference modality
model_effnet.trainable = False

model_effnet.summary()


### Inference Roberta

In [ ]:
def predict_roberta(model, tokenizer, texts, task_type="binary", batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    model.eval()

    all_probs = []
    all_labels = []

    for i in tqdm(range(0, len(texts), batch_size), desc=f"Inferenza RoBERTa ({task_type})"):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits

            if task_type == "binary":
                probs = torch.sigmoid(logits).squeeze().cpu().numpy()
                labels = (probs > 0.5).astype(int)

            elif task_type == "multiclass":
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                labels = np.argmax(probs, axis=1)

            else:
                raise ValueError("task_type deve essere 'binary' o 'multiclass'")

        all_probs.append(probs)
        all_labels.append(labels)

    # Concatenate
    pred_probs = np.concatenate(all_probs)
    pred_labels = np.concatenate(all_labels)

    return pred_probs, pred_labels

### Run Inference on Text Modality
#### Use RoBERTa models to generate predictions for both tasks.


In [ ]:
# Text Preparation
texts = df_test["cleaned_tweet_text"].tolist()

# Task 1
probs_text_task1, labels_text_task1 = predict_roberta(
    model=model_roberta_task1,
    tokenizer=tokenizer_roberta_task1,
    texts=texts,
    task_type="binary",
    batch_size=32
)

# Task 2
probs_text_task2, labels_text_task2 = predict_roberta(
    model=model_roberta_task2,
    tokenizer=tokenizer_roberta_task2,
    texts=texts,
    task_type="multiclass",
    batch_size=32
)


### Check and modify RoBERTa model's Prediction

In [ ]:
print("probs_text_task1 shape:", np.array(probs_text_task1).shape)

print("\nPredictions on task 1")
print(probs_text_task1[0])
print(labels_text_task1[0])

print("\n Predictions on task 2")
print(probs_text_task2[0])
print(labels_text_task2[0])

In [ ]:
probs_text_task1 = probs_text_task1[:, 1]  # use only the class 1 probability
print("probs_text_task1 shape:", np.array(probs_text_task1).shape)

### Run Inference on Image Modality
#### Use EfficientNetB0 models to generate predictions for both tasks.

In [ ]:
def predict_effnet(model, df, image_column="image_path", image_size=(224, 224), batch_size=32):
    paths = df[image_column].tolist()
    probs_task1 = []
    probs_task2 = []

    for i in tqdm(range(0, len(paths), batch_size), desc="Inferenza EfficientNet"):
        batch_paths = paths[i:i+batch_size]
        batch_images = []

        for path in batch_paths:
            try:
                img = image.load_img(os.path.join(dataset_dir, path), target_size=image_size)
                img_array = image.img_to_array(img)
                img_array = preprocess_input(img_array)
                batch_images.append(img_array)
            except Exception as e:
                print(f"Errore immagine {path}: {e}")
                batch_images.append(np.zeros((*image_size, 3)))

        batch_images = np.array(batch_images)
        p_task1, p_task2 = model.predict(batch_images, verbose=0)
        probs_task1.extend(p_task1.squeeze())
        probs_task2.extend(p_task2)

    # Task 1: binario
    labels_task1 = (np.array(probs_task1) > 0.5).astype(int)

    # Task 2: multiclass
    labels_task2 = np.argmax(np.array(probs_task2), axis=1)

    return np.array(probs_task1), labels_task1, np.array(probs_task2), labels_task2


In [ ]:
probs_img_task1, labels_img_task1, probs_img_task2, labels_img_task2 = predict_effnet(
    model=model_effnet,
    df=df_test,
    image_column="image_path"
)

### Check EfficientNetb0 model's Prediction

In [ ]:
print("probs_img_task1 shape :", np.array(probs_img_task1).shape)

print("\nPredictions on task 1")
print(probs_img_task1[0])
print(labels_img_task1[0])

print("\n Predictions on task 2")
print(probs_img_task2[0])
print(labels_img_task2[0])

# Class Mapping in the labelEconder
print("Label mapping:", dict(zip(le_info.classes_, le_info.transform(le_info.classes_))))

### Late Fusion Function
#### Combine predictions from text and image modalities with optimized weighted averaging.

In [ ]:
def optimize_alpha(probs_text, probs_image, y_true, task_type="binary"):
    """
    Cerca il miglior valore di alpha per la fusione tardi tra due vettori di probabilità.
    
    Args:
        probs_text (np.array): probabilità dal modello testuale (shape: [n_samples, n_classes])
        probs_image (np.array): probabilità dal modello visivo
        y_true (np.array): etichette reali
        task_type (str): "binary" o "multiclass"
        
    Returns:
        best_alpha (float): valore di alpha che ottimizza l'accuracy
        best_accuracy (float): accuracy ottenuta col miglior alpha
    """
    best_alpha = 0.0
    best_accuracy = 0.0

    for alpha in np.arange(0.0, 1.01, 0.05):
        fused_probs = alpha * probs_text + (1 - alpha) * probs_image
        
        if task_type == "binary":
            preds = np.round(fused_probs).astype(int)
        elif task_type == "multiclass":
            preds = np.argmax(fused_probs, axis=1)
        else:
            raise ValueError("task_type must be 'binary' or 'multiclass'")

        acc = accuracy_score(y_true, preds)
        if acc > best_accuracy:
            best_accuracy = acc
            best_alpha = alpha

    return best_alpha


In [ ]:
# Task info (binary)
best_alpha_info = optimize_alpha(probs_text_task1, probs_img_task1, y_test_task1, task_type="binary")
print(f"Best alpha for Task 1 (Informative): {best_alpha_info}")

# Task human (multiclass)
best_alpha_human = optimize_alpha(probs_text_task2, probs_img_task2, y_test_task2, task_type="multiclass")
print(f"Best alpha for Task 2 (Humanitarian): {best_alpha_human}")


### Run Late Fusion and Evaluate
#### Compute weighted prediction fusion and print performance for both tasks.

In [ ]:
def plot_reliability_curve(y_true, y_prob, task_name):
    """
    Plots a reliability diagram (calibration curve) for predicted probabilities.

    Args:
        y_true (array-like): True binary or multiclass labels.
        y_prob (array-like): Predicted probabilities for the positive class or each class.
        task_name (str): Name of the task for plot title.
    """
    # Compute calibration curve (fraction of positives vs. mean predicted value)
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10)
    plt.figure(figsize=(6, 5))
    plt.plot(prob_pred, prob_true, marker='o', label='Model')
    plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly calibrated')
    plt.title(f"{task_name} - Reliability Diagram")
    plt.xlabel("Predicted probability")
    plt.ylabel("True probability")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_roc_auc(y_true, y_scores, class_names, task_name):
	"""
	Plots ROC curve(s) and computes AUC for binary or multiclass classification.

	Args:
		y_true (array-like): True labels.
		y_scores (array-like): Predicted probabilities or scores.
		class_names (list): List of class names for multiclass.
		task_name (str): Name of the task for plot title.
	"""
	if len(np.unique(y_true)) == 2:
		# Binary classification ROC curve
		roc_auc = roc_auc_score(y_true, y_scores)
		fpr, tpr, _ = roc_curve(y_true, y_scores)

		plt.figure(figsize=(6, 5))
		plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.2f})")
		plt.plot([0, 1], [0, 1], "k--")
		plt.xlabel("False Positive Rate")
		plt.ylabel("True Positive Rate")
		plt.title(f"{task_name} - ROC Curve")
		plt.legend(loc="lower right")
		plt.grid(True)
		plt.tight_layout()
		plt.show()

	else:
		# Multiclass ROC curve (one-vs-rest for each class)
		n_classes = len(class_names)
		y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))

		fpr = {}
		tpr = {}
		roc_auc = {}

		for i in range(n_classes):
			fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
			roc_auc[i] = auc(fpr[i], tpr[i])

		colors = cycle(["aqua", "darkorange", "cornflowerblue", "green", "red"])
		plt.figure(figsize=(8, 6))

		for i, color in zip(range(n_classes), colors):
			plt.plot(fpr[i], tpr[i], color=color, lw=2,
					 label=f"Class {class_names[i]} (AUC = {roc_auc[i]:.2f})")
		
		plt.plot([0, 1], [0, 1], "k--", lw=2)
		plt.xlim([0.0, 1.0])
		plt.ylim([0.0, 1.05])
		plt.xlabel("False Positive Rate")
		plt.ylabel("True Positive Rate")
		plt.title(f"{task_name} - ROC Curve")
		plt.legend(loc="lower right")
		plt.grid(True)
		plt.tight_layout()
		plt.show()

In [ ]:
def evaluate_task(y_true, y_pred, task_name, class_names=None, y_scores=None):
    """
    Evaluates classification performance for a given task, printing a classification report,
    plotting a confusion matrix, and (optionally) ROC-AUC and reliability diagrams.

    Args:
        y_true (array-like): True labels.
        y_pred (array-like): Predicted labels.
        task_name (str): Name of the task (for plot/report titles).
        class_names (list, optional): List of class names for display.
        y_scores (array-like, optional): Predicted probabilities or scores for ROC/reliability plots.
    """
    # Print the classification report with precision, recall, f1-score, and support
    print(f"\n{task_name} - Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # Prepare display names for confusion matrix (capitalize for Task #2)
    if task_name == "Task #2":
        display_names = [c[0].upper() for c in class_names]
    else:
        display_names = class_names

    # Compute and plot the confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=display_names, yticklabels=display_names)
    plt.title(f"{task_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()
 
    # If probability scores are provided, plot ROC-AUC and reliability diagrams
    if y_scores is not None:
        plot_roc_auc(y_true, y_scores, class_names, task_name)
        # Only plot reliability diagram for binary tasks
        if len(np.unique(y_true)) == 2:
            plot_reliability_curve(y_true, y_scores, task_name)

In [ ]:
def late_fusion_predict(probs_text, probs_img, true_labels, task_type="binary", alpha=0.5, task_name="", class_names=None):
    """
    Perform late fusion prediction using weighted average of text and image probabilities,
    then evaluate with ROC-AUC, reliability diagram, confusion matrix, classification report.

    Args:
        probs_text (np.ndarray): Probabilities from text model.
        probs_img (np.ndarray): Probabilities from image model.
        true_labels (np.ndarray): Ground truth labels.
        task_type (str): "binary" or "multiclass".
        alpha (float): Weight to assign to text predictions (0.0 to 1.0).
        task_name (str): Task name for plot titles.
        class_names (list, optional): Class labels for confusion matrix & report.

    Returns:
        np.ndarray: Predicted labels.
    """
    fused_probs = alpha * probs_text + (1 - alpha) * probs_img

    if task_type == "binary":
        pred_labels = (fused_probs > 0.5).astype(int)
    elif task_type == "multiclass":
        pred_labels = np.argmax(fused_probs, axis=1)
    else:
        raise ValueError("task_type must be 'binary' or 'multiclass'")

    # Evaluation with plots and metrics
    evaluate_task(
        y_true=true_labels,
        y_pred=pred_labels,
        task_name=task_name,
        class_names=class_names,
        y_scores=fused_probs
    )

    return pred_labels


In [ ]:
# Task 1 (Binary)
late_fusion_predict(
    probs_text=probs_text_task1,
    probs_img=probs_img_task1,
    true_labels=y_test_task1,
    task_type="binary",
    alpha=best_alpha_info,
    task_name="Task #1",
    class_names=["Not Informative", "Informative"]
)

# Task 2 (Multiclass)
late_fusion_predict(
    probs_text=probs_text_task2,
    probs_img=probs_img_task2,
    true_labels=y_test_task2,
    task_type="multiclass",
    alpha=best_alpha_human,
    task_name="Task #2",
    class_names=["affected_individuals", "infrastructure_and_utility_damage", "not_humanitarian", "other_relevant_information", "rescue_volunteering_or_donation_effort"]
)

